# 0. Libraries

In [9]:
import json
import pandas as pd
from utilities.extract_qids import extract_qids

from utilities.alignment import (
    align_sections,
    align_sections_sentence_best_median,
    display_alignment,
    save_alignment,
)

from utilities.load_models import load_Qwen3_8B_via_llama_cpp
from utilities.omission_3v2 import calculate_omission,load_qwen_embedding
from utilities.propositioner_3v2 import extract_propositions_V3
from utilities.translate_json import translate_json, replace_ap_es_text


# 1. Wikipedia content download

In [10]:

data_en = extract_qids(
    "en",
    "Falkland_Islands",
    verbose=False,
) 
# data_en and data_es are lists of dictionaries
data_es = extract_qids(
    "es",
    "Islas_Malvinas_(territorio_británico_de_ultramar)",
    verbose=False,
)

with open( "data_es.json", "w", encoding="utf-8" ) as f:
    json.dump(data_es, f, indent=4, ensure_ascii=False)
    print("Triples saved to data_es.json")

with open( "data_en.json", "w", encoding="utf-8" ) as f:
    json.dump(data_en, f, indent=4, ensure_ascii=False)
    print("Triples saved to data_en.json")




   batch 1/11 (25 titles)
   batch 2/11 (25 titles)
   batch 3/11 (25 titles)
   batch 4/11 (25 titles)
   batch 5/11 (25 titles)
   batch 6/11 (25 titles)
   batch 7/11 (25 titles)
   batch 8/11 (25 titles)
   batch 9/11 (25 titles)
   batch 10/11 (25 titles)
   batch 11/11 (14 titles)
   batch 1/11 (25 titles)
   batch 2/11 (25 titles)
   batch 3/11 (25 titles)
   batch 4/11 (25 titles)
   batch 5/11 (25 titles)
   batch 6/11 (25 titles)
   batch 7/11 (25 titles)
   batch 8/11 (25 titles)
   batch 9/11 (25 titles)
   batch 10/11 (25 titles)
   batch 11/11 (14 titles)
   batch 1/7 (25 titles)
   batch 2/7 (25 titles)
   batch 3/7 (25 titles)
   batch 4/7 (25 titles)
   batch 5/7 (25 titles)
   batch 6/7 (25 titles)
   batch 7/7 (8 titles)
   batch 1/7 (25 titles)
   batch 2/7 (25 titles)
   batch 3/7 (25 titles)
   batch 4/7 (25 titles)
   batch 5/7 (25 titles)
   batch 6/7 (25 titles)
   batch 7/7 (8 titles)
Triples saved to data_es.json
Triples saved to data_en.json


# 2. Section alignment

In [11]:
# ALIGNMENT METHOD
# Choose "manual" or "sentence_best_median".
ALIGNMENT_METHOD = "sentence_best_median"

MIN_SCORE = 0.80

if ALIGNMENT_METHOD == "manual":
    alignment_results = align_sections(
        article_a=data_es,
        article_b=data_en,
        min_score=MIN_SCORE,
    )
    output_path = (
        "results/"
        "falkland_malvinas_alignment_manual.json"
    )

elif ALIGNMENT_METHOD == "sentence_best_median":
    alignment_results = align_sections_sentence_best_median(
        article_a=data_es,
        article_b=data_en,
        min_score=MIN_SCORE,
    )
    output_path = (
        "results/"
        "falkland_malvinas_alignment_best_sentence_median.json"
    )

else:
    raise ValueError(
        f"Unknown alignment method: {ALIGNMENT_METHOD!r}"
    )

display_alignment(alignment_results)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 14887.09it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 12840.77it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEX


SECTION ALIGNMENTS
es:Q9648:0 <-> en:Q9648:0  es:Q9648:0+en:Q9648:0+Best Sentence Median  Best Sentence Median  score=0.8626
es:Q9648:1 <-> en:Q9648:1  es:Q9648:1+en:Q9648:1+Best Sentence Median  Best Sentence Median  score=0.8829
es:Q9648:2 <-> en:Q9648:2  es:Q9648:2+en:Q9648:2+Best Sentence Median  Best Sentence Median  score=0.8746
es:Q9648:3 <-> en:Q9648:3  es:Q9648:3+en:Q9648:3+Best Sentence Median  Best Sentence Median  score=0.8727
es:Q9648:4 <-> en:Q9648:5  es:Q9648:4+en:Q9648:5+Best Sentence Median  Best Sentence Median  score=0.8555
es:Q9648:5 <-> en:Q9648:6  es:Q9648:5+en:Q9648:6+Best Sentence Median  Best Sentence Median  score=0.8596
es:Q9648:6 <-> en:Q9648:7  es:Q9648:6+en:Q9648:7+Best Sentence Median  Best Sentence Median  score=0.9225
es:Q9648:7 <-> en:Q9648:9  es:Q9648:7+en:Q9648:9+Best Sentence Median  Best Sentence Median  score=0.8581


In [12]:
alignment_results

[{'section_id_a': 'es:Q9648:0',
  'section_id_b': 'en:Q9648:0',
  'pair_id': 'es:Q9648:0+en:Q9648:0+Best Sentence Median',
  'alignment_score': 'Best Sentence Median',
  'numeric_alignment_score': 0.8626},
 {'section_id_a': 'es:Q9648:1',
  'section_id_b': 'en:Q9648:1',
  'pair_id': 'es:Q9648:1+en:Q9648:1+Best Sentence Median',
  'alignment_score': 'Best Sentence Median',
  'numeric_alignment_score': 0.8829},
 {'section_id_a': 'es:Q9648:2',
  'section_id_b': 'en:Q9648:2',
  'pair_id': 'es:Q9648:2+en:Q9648:2+Best Sentence Median',
  'alignment_score': 'Best Sentence Median',
  'numeric_alignment_score': 0.8746},
 {'section_id_a': 'es:Q9648:3',
  'section_id_b': 'en:Q9648:3',
  'pair_id': 'es:Q9648:3+en:Q9648:3+Best Sentence Median',
  'alignment_score': 'Best Sentence Median',
  'numeric_alignment_score': 0.8727},
 {'section_id_a': 'es:Q9648:4',
  'section_id_b': 'en:Q9648:5',
  'pair_id': 'es:Q9648:4+en:Q9648:5+Best Sentence Median',
  'alignment_score': 'Best Sentence Median',
  'numer

# 3. Proposition extraction

In [15]:
qwen_model = load_Qwen3_8B_via_llama_cpp()
# Extract the two articles

In [8]:
propositions_list_en_qwen = []

for section in data_en["sections"]:
    propositions=extract_propositions_V3(
                section=section,
                model=qwen_model,
                language=section["language"]
            )

        # extend instead of append so that the final JSON
        # contains one flat list of propositions.

    propositions_list_en_qwen.extend(propositions)

    # -----------------------------------------------------------------
    # Save JSON
    # -----------------------------------------------------------------

with open("propositions_en_sentence_level_qwen.json", "w", encoding="utf-8") as file:
    json.dump( propositions_list_en_qwen, file, indent=4, ensure_ascii=False)

print("-----------------------------------")
print("Atomization completed")
print(f"Total propositions: {len(propositions_list_en_qwen)}")
print(f"Saved to: propositions_en_sentence_level_qwen.json")
print("-----------------------------------")



Processing section: en:Q9648:0
Number of sentences: 20
Processing sentence 1/20
Propositions for sentence: 3
Processing sentence 2/20
Propositions for sentence: 3
Processing sentence 3/20
Propositions for sentence: 5
Processing sentence 4/20
Propositions for sentence: 4
Processing sentence 5/20
Propositions for sentence: 3
Processing sentence 6/20
Propositions for sentence: 2
Processing sentence 7/20
Propositions for sentence: 2
Processing sentence 8/20
Propositions for sentence: 4
Processing sentence 9/20
Propositions for sentence: 2
Processing sentence 10/20
Propositions for sentence: 1
Processing sentence 11/20
Propositions for sentence: 2
Processing sentence 12/20
Propositions for sentence: 1
Processing sentence 13/20
Propositions for sentence: 2
Processing sentence 14/20
Propositions for sentence: 4
Processing sentence 15/20
Propositions for sentence: 2
Processing sentence 16/20
Propositions for sentence: 1
Processing sentence 17/20
Propositions for sentence: 2
Processing sentenc

In [9]:
propositions_list_es_qwen = []

for section in data_es["sections"]:
    propositions=extract_propositions_V3(
                section=section,
                model=qwen_model,
                language=section["language"]
            )

        # extend instead of append so that the final JSON
        # contains one flat list of propositions.

    propositions_list_es_qwen.extend(propositions)

    # -----------------------------------------------------------------
    # Save JSON
    # -----------------------------------------------------------------

with open("propositions_es_sentence_level_qwen.json", "w", encoding="utf-8") as file:
    json.dump( propositions_list_es_qwen, file, indent=4, ensure_ascii=False)

print("-----------------------------------")
print("Atomization completed")
print(f"Total propositions: {len(propositions_list_es_qwen)}")
print(f"Saved to: propositions_es_sentence_level_qwen.json")
print("-----------------------------------")


Processing section: es:Q9648:0
Number of sentences: 7
Processing sentence 1/7
Propositions for sentence: 5
Processing sentence 2/7
Propositions for sentence: 3
Processing sentence 3/7
Propositions for sentence: 2
Processing sentence 4/7
Propositions for sentence: 2
Processing sentence 5/7
Propositions for sentence: 2
Processing sentence 6/7
Propositions for sentence: 4
Processing sentence 7/7
Propositions for sentence: 3
Total propositions: 21

Processing section: es:Q9648:1
Number of sentences: 5
Processing sentence 1/5
Propositions for sentence: 2
Processing sentence 2/5
Propositions for sentence: 2
Processing sentence 3/5
Propositions for sentence: 3
Processing sentence 4/5
Propositions for sentence: 4
Processing sentence 5/5
Propositions for sentence: 0
Total propositions: 11

Processing section: es:Q9648:2
Number of sentences: 38
Processing sentence 1/38
Propositions for sentence: 1
Processing sentence 2/38
Propositions for sentence: 2
Processing sentence 3/38
Propositions for se

In [13]:
with open("propositions_en_sentence_level_qwen.json", "r", encoding="utf-8") as file:
    propositions_list_en_qwen = json.load(file)

with open("propositions_es_sentence_level_qwen.json", "r", encoding="utf-8") as file:
    propositions_list_es_qwen = json.load(file)
    propositions_list_es_qwen=translate_json(propositions_list_es_qwen)

Translating: 100%|██████████| 19/19 [00:17<00:00,  1.11it/s]


# 4. Omission Analysis

In [10]:
# manual
list_manual_section_pairs = [
    {
        "section_id_a": "es:Q9648:0",
        "section_id_b": "en:Q9648:0",
        "pair_id": "es:Q9648:0+en:Q9648:0+Manual",
        "alignment_score": "Manual"
    },
    {
        "section_id_a": "es:Q9648:1",
        "section_id_b": "en:Q9648:1",
        "pair_id": "es:Q9648:1+en:Q9648:1+Manual",
        "alignment_score": "Manual"
    },
    {
        "section_id_a": "es:Q9648:2",
        "section_id_b": "en:Q9648:2",
        "pair_id": "es:Q9648:2+en:Q9648:2+Manual",
        "alignment_score": "Manual"
    },
    {
        "section_id_a": "es:Q9648:3",
        "section_id_b": "en:Q9648:3",
        "pair_id": "es:Q9648:3+en:Q9648:3+Manual",
        "alignment_score": "Manual"
    },
    {
        "section_id_a": "es:Q9648:5",
        "section_id_b": "en:Q9648:6",
        "pair_id": "es:Q9648:5+en:Q9648:6+Manual",
        "alignment_score": "Manual"
    },
    {
        "section_id_a": "es:Q9648:6",
        "section_id_b": "en:Q9648:7",
        "pair_id": "es:Q9648:6+en:Q9648:7+Manual",
        "alignment_score": "Manual"
    }
]

In [17]:
tokenizer,model_embedding,device=load_qwen_embedding()

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 5087.32it/s]


In [18]:


results_omission=[]

for pair in alignment_results:
    section_pair_id=pair["pair_id"]
    section_id_en=pair["section_id_b"]
    section_id_es=pair["section_id_a"]

    section_aps_en=[
        {
            "proposition_id":ap["proposition_id"],
            "text":ap["text"]
        }
        for ap in propositions_list_en_qwen
        if section_id_en == ap["section_id"]
    ]

    section_aps_es=[
        {
            "proposition_id":ap["proposition_id"],
            "text":ap["text_en"]
        }
        for ap in propositions_list_es_qwen
        if section_id_es == ap["section_id"]
    ]

    print(
        f"{section_id_en} (EN) vs {section_id_es} (ES) | "
        f"EN: {len(section_aps_en)} | ES: {len(section_aps_es)}"
    )

    omission_metrics=calculate_omission(
        section_aps_en,
        section_aps_es,
        threshold=0.70,
        tokenizer=tokenizer,
        model=model_embedding,
        device=device
    )

    results_omission.append({
        "section_pair_id":section_pair_id,
        "section_en":{
            "section_id":section_id_en,
            "propositions":section_aps_en
        },
        "section_es":{
            "section_id":section_id_es,
            "propositions":section_aps_es
        },
        "omission_metrics":omission_metrics
    })

with open("omission_results_qwen.json","w",encoding="utf-8") as file:
    json.dump(
        results_omission,
        file,
        ensure_ascii=False,
        indent=4
    )


en:Q9648:0 (EN) vs es:Q9648:0 (ES) | EN: 53 | ES: 21
en:Q9648:1 (EN) vs es:Q9648:1 (ES) | EN: 28 | ES: 11
en:Q9648:2 (EN) vs es:Q9648:2 (ES) | EN: 147 | ES: 92
en:Q9648:3 (EN) vs es:Q9648:3 (ES) | EN: 61 | ES: 45
en:Q9648:5 (EN) vs es:Q9648:4 (ES) | EN: 37 | ES: 29
en:Q9648:6 (EN) vs es:Q9648:5 (ES) | EN: 67 | ES: 19
en:Q9648:7 (EN) vs es:Q9648:6 (ES) | EN: 47 | ES: 39
en:Q9648:9 (EN) vs es:Q9648:7 (ES) | EN: 23 | ES: 44


# 5. Non-omitted AP equivalents

In [19]:
equivalent_aps = {}


for section in results_omission:
    section_pair_id = section["section_pair_id"]
    metrics = section["omission_metrics"]

    # ---------------------------------------------------------
    # EN -> ES
    # ---------------------------------------------------------

    for match_group in metrics["en_to_es"]["matches"]:
        ap_en = match_group["ap"]

        for match in match_group["matches"]:
            ap_es = match["ap"]

            key = (
                section_pair_id,
                ap_en["proposition_id"],
                ap_es["proposition_id"]
            )

            equivalent_aps[key] = {
                "section_pair_id": section_pair_id,
                "ap_en_id": ap_en["proposition_id"],
                "ap_en_text": ap_en["text"],
                "ap_es_id": ap_es["proposition_id"],
                "ap_es_text": ap_es["text"],
                "similarity": match["similarity"]
            }


    # ---------------------------------------------------------
    # ES -> EN
    # ---------------------------------------------------------

    for match_group in metrics["es_to_en"]["matches"]:
        ap_es = match_group["ap"]

        for match in match_group["matches"]:
            ap_en = match["ap"]

            key = (
                section_pair_id,
                ap_en["proposition_id"],
                ap_es["proposition_id"]
            )

            # Solo añadir si no estaba ya en EN -> ES
            if key not in equivalent_aps:
                equivalent_aps[key] = {
                    "section_pair_id": section_pair_id,
                    "ap_en_id": ap_en["proposition_id"],
                    "ap_en_text": ap_en["text"],
                    "ap_es_id": ap_es["proposition_id"],
                    "ap_es_text": ap_es["text"],
                    "similarity": match["similarity"]
                }


# Convertir a lista
equivalent_aps = list(equivalent_aps.values())
equivalent_aps = replace_ap_es_text(equivalent_aps,propositions_list_es_qwen)

with open("equivalent_aps_qwen.json", "w", encoding="utf-8") as f:
    json.dump(equivalent_aps, f, ensure_ascii=False, indent=4)

# 6. LLM metrics prediction

In [20]:
%pip install huggingface_hub

python(89988) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Note: you may need to restart the kernel to use updated packages.


In [21]:
import importlib
from pathlib import Path

from huggingface_hub import hf_hub_download
from llama_cpp import Llama
from IPython.display import display
import llm_metrics_prediction as metrics

# Pick up the latest script changes.
metrics = importlib.reload(metrics)

INPUT_PATH = Path("equivalent_aps_qwen.json")
MODEL_PATH = Path("models/Qwen3-8B-Q4_K_M.gguf")
MODEL_REPO = "Qwen/Qwen3-8B-GGUF"
OUTPUT_CSV = Path("llm_metrics_predictions_qwen.csv")
OUTPUT_JSON = Path("llm_metrics_predictions_qwen.json")

# Load proposition pairs directly from the JSON structure.
df_pairs = metrics.load_input(str(INPUT_PATH))
print(f"Loaded {len(df_pairs)} proposition pairs.")

# Download the GGUF model only if the local file is missing or empty.
if not MODEL_PATH.is_file() or MODEL_PATH.stat().st_size == 0:
    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {MODEL_PATH.name}...")
    downloaded_path = hf_hub_download(
        repo_id=MODEL_REPO,
        filename=MODEL_PATH.name,
        local_dir=str(MODEL_PATH.parent),
        force_download=MODEL_PATH.is_file(),
    )
    MODEL_PATH = Path(downloaded_path)

# Reload the model so the new llama.cpp settings take effect.
if isinstance(globals().get("qwen_model"), Llama):
    qwen_model.close()
    qwen_model = None

print("Loading Qwen with llama.cpp...")
qwen_model = Llama(
    model_path=str(MODEL_PATH.resolve()),
    n_ctx=4096,
    n_gpu_layers=-1,
    offload_kqv=True,
    flash_attn=True,
    n_batch=512,
    n_ubatch=512,
    verbose=True,
)

# Predict all five metrics; similarity is excluded from predictions.
predictions_qwen = metrics.classify_dataframe(df_pairs, qwen_model)

metrics.save_output(
    predictions_qwen,
    csv_path=str(OUTPUT_CSV),
    json_path=str(OUTPUT_JSON),
)

print(f"Saved CSV: {OUTPUT_CSV.resolve()}")
print(f"Saved JSON: {OUTPUT_JSON.resolve()}")
display(predictions_qwen.head())


Loaded 139 proposition pairs.
Loading Qwen with llama.cpp...


llama_model_loader: loaded meta data with 28 key-value pairs and 399 tensors from /Users/fserracrespi/Desktop/historically-disputed-events-online-encyclopedias/models/Qwen3-8B-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen3 8B Awq Compatible Instruct
llama_model_loader: - kv   3:                           general.finetune str              = awq-compatible-Instruct
llama_model_loader: - kv   4:                           general.basename str              = Qwen3
llama_model_loader: - kv   5:                         general.size_label str              = 8B
llama_model_loader: - kv   6:                          qw

1/139 | pair_id=1 | comparable=Yes | contradiction=No | agency=0 | causality=0 | language=0


llama_perf_context_print:        load time =    3777.12 ms
llama_perf_context_print: prompt eval time =    3465.54 ms /   860 tokens (    4.03 ms per token,   248.16 tokens per second)
llama_perf_context_print:        eval time =     249.07 ms /     6 runs   (   41.51 ms per token,    24.09 tokens per second)
llama_perf_context_print:       total time =    3847.74 ms /   866 tokens
llama_perf_context_print:    graphs reused =          5
Llama.generate: 14 prefix-match hit, remaining 915 prompt tokens to eval
llama_perf_context_print:        load time =    3777.12 ms
llama_perf_context_print: prompt eval time =    3780.06 ms /   915 tokens (    4.13 ms per token,   242.06 tokens per second)
llama_perf_context_print:        eval time =     245.23 ms /     6 runs   (   40.87 ms per token,    24.47 tokens per second)
llama_perf_context_print:       total time =    4146.36 ms /   921 tokens
llama_perf_context_print:    graphs reused =          5
Llama.generate: 8 prefix-match hit, remaining

2/139 | pair_id=2 | comparable=Yes | contradiction=No | agency=0 | causality=0 | language=0


...

llama_perf_context_print:        load time =    3777.12 ms
llama_perf_context_print: prompt eval time =    3501.03 ms /   861 tokens (    4.07 ms per token,   245.93 tokens per second)
llama_perf_context_print:        eval time =     248.49 ms /     6 runs   (   41.41 ms per token,    24.15 tokens per second)
llama_perf_context_print:       total time =    3869.68 ms /   867 tokens
llama_perf_context_print:    graphs reused =          5
Llama.generate: 14 prefix-match hit, remaining 916 prompt tokens to eval
llama_perf_context_print:        load time =    3777.12 ms
llama_perf_context_print: prompt eval time =    3755.65 ms /   916 tokens (    4.10 ms per token,   243.90 tokens per second)
llama_perf_context_print:        eval time =     247.85 ms /     6 runs   (   41.31 ms per token,    24.21 tokens per second)
llama_perf_context_print:       total time =    4107.25 ms /   922 tokens
llama_perf_context_print:    graphs reused =          5
Llama.generate: 8 prefix-match hit, remaining

138/139 | pair_id=138 | comparable=No | contradiction=Yes | agency=2 | causality=2 | language=2


llama_perf_context_print:        load time =    3777.12 ms
llama_perf_context_print: prompt eval time =    3488.99 ms /   854 tokens (    4.09 ms per token,   244.77 tokens per second)
llama_perf_context_print:        eval time =     247.07 ms /     6 runs   (   41.18 ms per token,    24.29 tokens per second)
llama_perf_context_print:       total time =    3852.06 ms /   860 tokens
llama_perf_context_print:    graphs reused =          5
Llama.generate: 14 prefix-match hit, remaining 909 prompt tokens to eval
llama_perf_context_print:        load time =    3777.12 ms
llama_perf_context_print: prompt eval time =    3753.31 ms /   909 tokens (    4.13 ms per token,   242.19 tokens per second)
llama_perf_context_print:        eval time =     246.93 ms /     6 runs   (   41.16 ms per token,    24.30 tokens per second)
llama_perf_context_print:       total time =    4119.44 ms /   915 tokens
llama_perf_context_print:    graphs reused =          5
Llama.generate: 8 prefix-match hit, remaining

139/139 | pair_id=139 | comparable=Yes | contradiction=Yes | agency=2 | causality=2 | language=2
Saved CSV: /Users/fserracrespi/Desktop/historically-disputed-events-online-encyclopedias/llm_metrics_predictions_qwen.csv
Saved JSON: /Users/fserracrespi/Desktop/historically-disputed-events-online-encyclopedias/llm_metrics_predictions_qwen.json


,pair_id,section_pair_id,ap_en_id,ap_en_text,ap_es_id,ap_es_text,model_id,comparable,contradiction,agency,causality,language,status,raw_responses
0,1,es:Q9648:0+en:Q9648:0+Best Sentence Median,en:Q9648:0:ap:0,The Falkland Islands is an archipelago in the ...,es:Q9648:0:ap:3,El archipiélago de las Malvinas está situado e...,qwen3-8b,Yes,No,0,0,0,ok,"{""comparable"": ""{\""label\"": \""Yes\""}"", ""contra..."
1,2,es:Q9648:0+en:Q9648:0+Best Sentence Median,en:Q9648:0:ap:0,The Falkland Islands is an archipelago in the ...,es:Q9648:0:ap:4,El archipiélago de las Malvinas está situado e...,qwen3-8b,Yes,No,0,0,0,ok,"{""comparable"": ""{\""label\"": \""Yes\""}"", ""contra..."
2,3,es:Q9648:0+en:Q9648:0+Best Sentence Median,en:Q9648:0:ap:1,The Falkland Islands is located on the Patagon...,es:Q9648:0:ap:3,El archipiélago de las Malvinas está situado e...,qwen3-8b,Yes,No,2,0,2,ok,"{""comparable"": ""{\""label\"": \""Yes\""}"", ""contra..."
3,4,es:Q9648:0+en:Q9648:0+Best Sentence Median,en:Q9648:0:ap:1,The Falkland Islands is located on the Patagon...,es:Q9648:0:ap:4,El archipiélago de las Malvinas está situado e...,qwen3-8b,Yes,No,0,0,1,ok,"{""comparable"": ""{\""label\"": \""Yes\""}"", ""contra..."
4,5,es:Q9648:0+en:Q9648:0+Best Sentence Median,en:Q9648:0:ap:2,The Falkland Islands is commonly referred to a...,es:Q9648:0:ap:3,El archipiélago de las Malvinas está situado e...,qwen3-8b,Yes,No,0,0,0,ok,"{""comparable"": ""{\""label\"": \""Yes\""}"", ""contra..."
